# The Economic DNA of Nations (1960–2025)
### A Deep-Dive into Global Economic Stress, Crises, and Convergence
*Developed for Kaggle — Analyze the trajectories, footprints, and warnings signs of economic collapse.*

---
**Core Dataset Files:**
- `country_metadata.csv` — Spatial and regional identifiers
- `country_year_indicators.csv` — Yearly macroeconomic and food indicators
- `economic_stress_score.csv` — Percentile stress indicators and composite score


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from scipy.spatial.distance import cdist

# Premium Theme Setup
plt.rcParams.update({
    'figure.facecolor': '#0d1117',
    'axes.facecolor': '#161b22',
    'axes.edgecolor': '#30363d',
    'axes.labelcolor': '#c9d1d9',
    'axes.titlecolor': '#e6edf3',
    'xtick.color': '#8b949e',
    'ytick.color': '#8b949e',
    'text.color': '#c9d1d9',
    'grid.color': '#21262d',
    'grid.alpha': 0.6,
    'legend.facecolor': '#161b22',
    'legend.edgecolor': '#30363d',
    'font.family': 'DejaVu Sans',
    'axes.titlesize': 14,
    'axes.labelsize': 11,
})

PALETTE = ['#58a6ff', '#3fb950', '#f78166', '#d2a8ff', '#ffa657', '#79c0ff', '#56d364', '#ff7b72']
print("Style & imports loaded successfully!")


In [ ]:
def load_dataset(filename):
    paths_to_try = [
        filename,
        os.path.join('Global Stress Indicator', filename),
        os.path.join('/kaggle/input/datasets/sateasinpedas/global-geo-economic-stress-indicators', filename),
        os.path.join('..', filename),
        os.path.join('../input/global-geo-economic-stress-indicators', filename)
    ]
    for p in paths_to_try:
        if os.path.exists(p):
            return pd.read_csv(p)
    return pd.read_csv(os.path.join('/kaggle/input/datasets/sateasinpedas/global-geo-economic-stress-indicators', filename))

# Load files
metadata = load_dataset('country_metadata.csv')
indicators = load_dataset('country_year_indicators.csv')
scores = load_dataset('economic_stress_score.csv')

# Merge
df = pd.merge(
    indicators,
    scores[['country_code', 'year', 'inflation_score', 'unemployment_score', 'gdp_growth_score', 
            'income_vulnerability_score', 'food_pressure_score', 'final_economic_stress_score', 'stress_category']],
    on=['country_code', 'year'],
    how='inner'
)

print(f"Loaded successfully! Joined Data Shape: {df.shape[0]:,} rows x {df.shape[1]} columns.")


---
## 1. 65 Years of Global Macroeconomic Evolution
Here, we track how global averages of macroeconomic health and food security have changed over the decades.


In [ ]:
# Aggregate yearly means
world_avg = df.groupby('year')[['gdp_growth', 'inflation', 'unemployment', 'food_production_index', 'final_economic_stress_score']].mean().reset_index()

fig, axes = plt.subplots(2, 2, figsize=(22, 14))
fig.patch.set_facecolor('#0d1117')

# GDP Growth
axes[0, 0].plot(world_avg['year'], world_avg['gdp_growth'], color='#58a6ff', lw=2.5)
axes[0, 0].set_title('Global Average GDP Growth (%)')
axes[0, 0].axhline(0, color='#8b949e', ls=':', lw=1)
axes[0, 0].grid(True)

# Inflation
axes[0, 1].plot(world_avg['year'], world_avg['inflation'], color='#f78166', lw=2.5)
axes[0, 1].set_title('Global Average Inflation (%)')
axes[0, 1].set_yscale('log')
axes[0, 1].grid(True)

# Food Production Index
axes[1, 0].plot(world_avg['year'], world_avg['food_production_index'], color='#3fb950', lw=2.5)
axes[1, 0].set_title('Global Food Production Index (2014-2016 = 100)')
axes[1, 0].grid(True)

# Final Economic Stress Score
axes[1, 1].plot(world_avg['year'], world_avg['final_economic_stress_score'], color='#ffd700', lw=3.0)
axes[1, 1].set_title('Global Economic Stress Score (0-100)')
axes[1, 1].grid(True)

plt.suptitle('Global Macroeconomic Evolution (1960–2025)', fontsize=18, color='#e6edf3', fontweight='bold', y=0.98)
plt.savefig('section1_global_trends.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()


---
## 2. Regional Stress Atlas & The Most Improved Nations
In this section, we analyze economic stress across different World Bank regions and rank the countries that improved the most (stress decreased) and deteriorated the most since 1990.


In [ ]:
# Regional averages over time
regional_stress = df.groupby(['year', 'region'])['final_economic_stress_score'].mean().unstack()

plt.figure(figsize=(22, 10))
for i, col in enumerate(regional_stress.columns):
    plt.plot(regional_stress.index, regional_stress[col], label=col, lw=2.2, alpha=0.9)
plt.plot(world_avg['year'], world_avg['final_economic_stress_score'], label='Global Avg', color='white', ls='--', lw=3.0)

plt.title('Economic Stress Score Trajectory by Region (1960–2025)', fontsize=16, pad=15)
plt.xlabel('Year')
plt.ylabel('Economic Stress Score')
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.grid(True)
plt.tight_layout()
plt.savefig('section2_regional_stress.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()

# Improvement/Deterioration analysis since 1990
df_1990 = df[df['year'] == 1990][['country_name', 'final_economic_stress_score']].rename(columns={'final_economic_stress_score': 'stress_1990'})
df_2024 = df[df['year'] == 2024][['country_name', 'final_economic_stress_score']].rename(columns={'final_economic_stress_score': 'stress_2024'})
change_df = pd.merge(df_1990, df_2024, on='country_name').dropna()
change_df['stress_change'] = change_df['stress_2024'] - change_df['stress_1990']

top_improved = change_df.nsmallest(10, 'stress_change')  # Largest decrease in stress
top_deteriorated = change_df.nlargest(10, 'stress_change')  # Largest increase in stress

fig, axes = plt.subplots(1, 2, figsize=(22, 8))
fig.patch.set_facecolor('#0d1117')

# Improved
axes[0].barh(top_improved['country_name'], -top_improved['stress_change'], color='#3fb950', alpha=0.85)
axes[0].set_title('Top 10 Most Improved Countries (Stress Reduction, 1990→2024)', fontsize=13)
axes[0].set_xlabel('Stress Score Decrease')
axes[0].grid(True, axis='x')

# Deteriorated
axes[1].barh(top_deteriorated['country_name'], top_deteriorated['stress_change'], color='#f78166', alpha=0.85)
axes[1].set_title('Top 10 Most Deteriorated Countries (Stress Increase, 1990→2024)', fontsize=13)
axes[1].set_xlabel('Stress Score Increase')
axes[1].grid(True, axis='x')

plt.tight_layout()
plt.savefig('section2_stress_changes.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()


---
## 3. Hidden Country Clusters (Economic Fingerprints)
By summarizing each country's historical averages across all indicators, we can identify "economic fingerprints." Using PCA and t-SNE, we project these fingerprints into a 2D space to uncover natural country groupings.


In [ ]:
# Create country profiles (mean of values over all years)
feature_cols = ['gdp_growth', 'inflation', 'unemployment', 'gdp_per_capita', 'population', 
                'food_production_index', 'cereal_yield', 'dietary_energy_supply_adequacy', 
                'final_economic_stress_score']

profiles = df.groupby(['country_code', 'country_name', 'region'])[feature_cols].mean().reset_index()

# Drop rows with many missing features and fill remaining with median
profiles_clean = profiles.dropna(subset=['final_economic_stress_score'])
for col in feature_cols:
    profiles_clean[col] = profiles_clean[col].fillna(profiles_clean[col].median())

X = profiles_clean[feature_cols].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# t-SNE Projection
tsne = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000)
coords_tsne = tsne.fit_transform(X_scaled)
profiles_clean['tsne_x'] = coords_tsne[:, 0]
profiles_clean['tsne_y'] = coords_tsne[:, 1]

plt.figure(figsize=(20, 12))
regions = profiles_clean['region'].unique()
colors = plt.cm.tab10(np.linspace(0, 1, len(regions)))

for r, color in zip(regions, colors):
    subset = profiles_clean[profiles_clean['region'] == r]
    plt.scatter(subset['tsne_x'], subset['tsne_y'], label=r, s=100, color=color, alpha=0.85, edgecolors='none')

# Annotate notable countries
notable = ['USA', 'CHN', 'DEU', 'SGP', 'VEN', 'ARG', 'KOR', 'ZAF', 'IND', 'AFG']
for code in notable:
    row = profiles_clean[profiles_clean['country_code'] == code]
    if not row.empty:
        plt.annotate(row.iloc[0]['country_name'], (row.iloc[0]['tsne_x'], row.iloc[0]['tsne_y']),
                     textcoords="offset points", xytext=(0,10), ha='center', fontweight='bold', color='#ffd700', fontsize=9)

plt.title('Country Economic Fingerprints (t-SNE Projection of Macro-Indicators)', fontsize=16, pad=15)
plt.legend(loc='lower left', bbox_to_anchor=(0.02, 0.02), fontsize=10)
plt.grid(True, alpha=0.3)
plt.savefig('section3_economic_fingerprints.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()


---
## 4. Predicting Economic Collapse: Early-Warning Crisis Signals
In this section, we analyze the warning signs preceding economic crises. We identify crisis events where inflation spikes abnormally (>50%), GDP growth crashes (< -5%), or unemployment skyrockets (>20%), and trace how the key indicators behave in the 5 years leading up to the collapse.


In [ ]:
# Find crisis years
crises = []
for country, group in df.groupby('country_name'):
    group = group.sort_values('year')
    for idx in range(len(group)):
        row = group.iloc[idx]
        is_inflation_crisis = row['inflation'] > 50
        is_gdp_crisis = row['gdp_growth'] < -7
        is_unemployment_crisis = row['unemployment'] > 22
        
        if is_inflation_crisis or is_gdp_crisis or is_unemployment_crisis:
            # We record a crisis
            crisis_year = row['year']
            crises.append({'country': country, 'crisis_year': crisis_year})

crises_df = pd.DataFrame(crises).drop_duplicates(subset=['country']).head(40) # Limit to first 40 countries for readability

# Build aligned timeline database
aligned_data = []
for _, row in crises_df.iterrows():
    c_df = df[df['country_name'] == row['country']]
    cy = row['crisis_year']
    for yr_offset in range(-5, 1):
        target_yr = cy + yr_offset
        y_row = c_df[c_df['year'] == target_yr]
        if not y_row.empty:
            aligned_data.append({
                'country': row['country'],
                'years_to_crisis': yr_offset,
                'inflation': y_row.iloc[0]['inflation'],
                'gdp_growth': y_row.iloc[0]['gdp_growth'],
                'unemployment': y_row.iloc[0]['unemployment'],
                'stress_score': y_row.iloc[0]['final_economic_stress_score']
            })

aligned_df = pd.DataFrame(aligned_data)
aligned_avg = aligned_df.groupby('years_to_crisis')[['inflation', 'gdp_growth', 'unemployment', 'stress_score']].median().reset_index()

fig, axes = plt.subplots(2, 2, figsize=(22, 14))
fig.patch.set_facecolor('#0d1117')

# GDP Growth before crisis
axes[0, 0].plot(aligned_avg['years_to_crisis'], aligned_avg['gdp_growth'], marker='o', color='#58a6ff', lw=3)
axes[0, 0].set_title('Median GDP Growth (%) leading to Crisis')
axes[0, 0].set_xlabel('Years to Crisis')
axes[0, 0].grid(True)

# Inflation before crisis
axes[0, 1].plot(aligned_avg['years_to_crisis'], aligned_avg['inflation'], marker='o', color='#f78166', lw=3)
axes[0, 1].set_title('Median Inflation (%) leading to Crisis')
axes[0, 1].set_xlabel('Years to Crisis')
axes[0, 1].grid(True)

# Unemployment before crisis
axes[1, 0].plot(aligned_avg['years_to_crisis'], aligned_avg['unemployment'], marker='o', color='#3fb950', lw=3)
axes[1, 0].set_title('Median Unemployment (%) leading to Crisis')
axes[1, 0].set_xlabel('Years to Crisis')
axes[1, 0].grid(True)

# Stress Score before crisis
axes[1, 1].plot(aligned_avg['years_to_crisis'], aligned_avg['stress_score'], marker='o', color='#ffd700', lw=3)
axes[1, 1].set_title('Median Stress Score leading to Crisis')
axes[1, 1].set_xlabel('Years to Crisis')
axes[1, 1].grid(True)

plt.suptitle('Crisis Warning Timeline: 5 Years Before Economic Collapse', fontsize=18, color='#e6edf3', fontweight='bold', y=0.98)
plt.savefig('section4_early_warning_signals.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()


---
## 5. Living Beyond Their Means? Expected vs Actual Performance
By fitting a regression model predicting expected GDP per capita based on population scale, agricultural land share, region, and income group, we can identify overperformers (countries with positive residuals) and underperformers.


In [ ]:
# Filter latest data
# Find the latest year with sufficient non-null data for the regression
regression_cols = ['country_name', 'gdp_per_capita', 'population', 'agricultural_land_pct', 'region', 'income_group']
latest_year = df['year'].max()
model_df = pd.DataFrame()
while latest_year >= df['year'].min():
    subset = df[df['year'] == latest_year][regression_cols].dropna()
    if len(subset) > 50:
        model_df = subset
        break
    latest_year -= 1
print(f'Using year {latest_year} for regression analysis (sample size: {len(model_df)}).')
# One-hot encoding
model_df_encoded = pd.get_dummies(model_df, columns=['region', 'income_group'], drop_first=True, dtype=int)

# Predict log GDP per capita to stabilize variance
X_reg = model_df_encoded.drop(columns=['country_name', 'gdp_per_capita'])
y_reg = np.log(model_df_encoded['gdp_per_capita'])

reg = LinearRegression()
reg.fit(X_reg, y_reg)
y_pred = reg.predict(X_reg)

model_df['Expected_GDP_per_Capita'] = np.exp(y_pred)
model_df['Overperformance'] = model_df['gdp_per_capita'] - model_df['Expected_GDP_per_Capita']

top_over = model_df.nlargest(10, 'Overperformance')
top_under = model_df.nsmallest(10, 'Overperformance')

fig, axes = plt.subplots(1, 2, figsize=(22, 8))
fig.patch.set_facecolor('#0d1117')

# Overperformers
axes[0].barh(top_over['country_name'], top_over['Overperformance'] / 1000, color='#3fb950', alpha=0.85)
axes[0].set_title('Top 10 GDP Overperformers (Actual vs Expected)', fontsize=13)
axes[0].set_xlabel('GDP overperformance (Thousands USD)')
axes[0].grid(True, axis='x')

# Underperformers
axes[1].barh(top_under['country_name'], -top_under['Overperformance'] / 1000, color='#f78166', alpha=0.85)
axes[1].set_title('Top 10 GDP Underperformers (Actual vs Expected)', fontsize=13)
axes[1].set_xlabel('GDP underperformance (Thousands USD)')
axes[1].grid(True, axis='x')

plt.suptitle('Living Beyond Means: Actual vs Structural Expected GDP per Capita', fontsize=16, color='#e6edf3', fontweight='bold')
plt.tight_layout()
plt.savefig('section5_expected_gdp_residuals.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()


---
## 6. Decadal Economic League Tables
We score countries on a composite performance score representing stable growth, low inflation, low unemployment, and high food/income security. Below are the top 10 best-performing countries for each historical decade.


In [ ]:
# Compute Performance Score: lower stress scores mean higher performance
df['Performance_Score'] = 100 - df['final_economic_stress_score']

# Define decades
def get_decade(year):
    return f"{(year // 10) * 10}s"
df['decade'] = df['year'].apply(get_decade)

# Calculate decadal averages
decadal_avg = df.groupby(['decade', 'country_name'])['Performance_Score'].mean().reset_index()

# Find top performers per decade
decades_to_plot = ['1960s', '1980s', '2000s', '2020s']
for dec in decades_to_plot:
    dec_df = decadal_avg[decadal_avg['decade'] == dec].nlargest(10, 'Performance_Score')
    print(f"\n--- Top 10 Best Countries of the {dec} ---")
    print(dec_df[['country_name', 'Performance_Score']].to_string(index=False))


---
## 7. Economic Tribe Network Visualization
To represent the economic connectivity/similarity of countries, we calculate the Euclidean distance between their normalized macro-profiles and connect each country to its top 3 nearest economic neighbors.


In [ ]:
# Standardize features
profiles_matrix = X_scaled

# Calculate pairwise distances
dists = cdist(profiles_matrix, profiles_matrix, metric='euclidean')

# Coordinates (using t-SNE coords as nodes layout)
pos_x = profiles_clean['tsne_x'].values
pos_y = profiles_clean['tsne_y'].values
names = profiles_clean['country_name'].values

plt.figure(figsize=(22, 14))

# Draw nearest-neighbor connections
for i in range(len(profiles_clean)):
    # Sort distances to find nearest neighbors (exclude self at index 0)
    nearest_indices = np.argsort(dists[i])[1:4]
    for idx in nearest_indices:
        plt.plot([pos_x[i], pos_x[idx]], [pos_y[i], pos_y[idx]], color='#30363d', lw=0.8, alpha=0.6, zorder=1)

# Draw nodes
plt.scatter(pos_x, pos_y, color='#58a6ff', s=50, edgecolors='#161b22', linewidth=0.5, zorder=2, alpha=0.85)

# Label top countries
for code in notable:
    idx = profiles_clean[profiles_clean['country_code'] == code].index
    if len(idx) > 0:
        pos_idx = idx[0]
        plt.text(pos_x[pos_idx], pos_y[pos_idx] + 0.8, names[pos_idx],
                 ha='center', fontweight='bold', color='#ffd700', fontsize=10, zorder=3)

plt.title('Economic Tribe Network: Connecting Similar Global Economies', fontsize=16, pad=15)
plt.axis('off')
plt.savefig('section7_tribe_network.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
